
# Load from NumPy arrays

Arrays use axis order ``(z, y, x)``. The mask is integer labels,
``0`` = background. One person and several people are each a
:class:`~habit.contracts.Cohort`.


One person. Change the two paths to your files, then keep the arrays.



In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import SimpleITK as sitk

from habit.contracts import ArrayImageRef, Cohort, Geometry, Subject
from habit.datasets import fetch_demo
from habit.viz import plot_numpy_ingest
from habit.voxel_features import RawVoxelFeatures

DATA = fetch_demo()
IMAGE = DATA / "images" / "subj001" / "LAP" / "WATER__WATER__Ax_Dyn_LAVA_Flex+C_Series0009.nrrd"
MASK = DATA / "masks" / "subj001" / "LAP" / "WATER__BH_Ax_LAVA_Flex_10min_Series0017_mask.nrrd"
sitk_image = sitk.ReadImage(str(IMAGE))
array = sitk.GetArrayFromImage(sitk_image)
mask = np.asarray(sitk.GetArrayFromImage(sitk.ReadImage(str(MASK))), dtype=np.int32)
geometry = Geometry.from_array(
    array.shape,
    spacing=tuple(sitk_image.GetSpacing()),
    origin=tuple(sitk_image.GetOrigin()),
    direction=tuple(sitk_image.GetDirection()),
)
np_subject = Subject(
    subject_id="subj001",
    images={"LAP": ArrayImageRef(array=array, geometry=geometry)},
    masks={"LAP": ArrayImageRef(array=mask, geometry=geometry)},
)
one = Cohort([np_subject], name="one")
print(one)
field = RawVoxelFeatures(modalities=["LAP"])(np_subject)
print(field.feature_frame().head())
field.feature_frame().head()

Visual summary of this route: the array itself (left, as a colour matrix)
becomes a plottable Subject (right, zoomed to the ROI).



In [ ]:
Path("out").mkdir(exist_ok=True)
fig = plot_numpy_ingest(
    np_subject.image("LAP"),
    roi_mask=np_subject.mask("LAP"),
)
fig.savefig("out/numpy_subject_ingest.png", dpi=150, bbox_inches="tight")
plt.show()

Several people. One Subject per person, then one cohort.



In [ ]:
IMAGE_2 = DATA / "images" / "subj002" / "LAP" / "012_WATERWATERAxDynLAVAFlexC.nrrd"
MASK_2 = DATA / "masks" / "subj002" / "LAP" / "016_WATERWATERBHAxLAVAFlex5min_mask.nrrd"
sitk_image_2 = sitk.ReadImage(str(IMAGE_2))
array_2 = sitk.GetArrayFromImage(sitk_image_2)
mask_2 = np.asarray(sitk.GetArrayFromImage(sitk.ReadImage(str(MASK_2))), dtype=np.int32)
geometry_2 = Geometry.from_array(
    array_2.shape,
    spacing=tuple(sitk_image_2.GetSpacing()),
    origin=tuple(sitk_image_2.GetOrigin()),
    direction=tuple(sitk_image_2.GetDirection()),
)
subject_2 = Subject(
    subject_id="subj002",
    images={"LAP": ArrayImageRef(array=array_2, geometry=geometry_2)},
    masks={"LAP": ArrayImageRef(array=mask_2, geometry=geometry_2)},
)
many = Cohort([np_subject, subject_2], name="many")
print(many)